# VitalDB Colab GPU full training

> **LEGACY EXPLORATORY NOTE:** This physiological-inclusive 40-run notebook is preserved for reproducibility only. It must not be used for final simulator-compatible feature selection.

This notebook runs the historical full group-retraining experiment on a Colab CUDA GPU: four feature conditions, two model families, and five paired seeds (40 runs). Every training command explicitly uses `--device cuda`; outputs, checkpoints, manifests, and validation summaries are written directly to Google Drive.

Here, `validation-only` does not mean smoke training or a reduced training set. Each run trains on the complete training split for up to 50 epochs with early stopping on the complete validation split. The flag only prevents premature held-out test evaluation while candidate models are being compared.

Run `notebooks/colab_gpu_setup.ipynb` successfully before this notebook. Full execution is locked by default and is never started merely by opening or running all cells.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path

REPO_URL = 'https://github.com/khyitty/VitalDB-Feature-Selection.git'
REPO_DIR = Path('/content/VitalDB-Feature-Selection')
DRIVE_PROJECT_ROOT = Path('/content/drive/MyDrive/VitalDB-Feature-Selection')
DATASET_DIR = DRIVE_PROJECT_ROOT / 'data/modeling/full'
OUTPUT_ROOT = DRIVE_PROJECT_ROOT / 'outputs'
SETUP_AUDIT_DIR = OUTPUT_ROOT / 'colab_environment'
GRU_SMOKE_DIR = OUTPUT_ROOT / 'colab_smoke/gru/seed_42'
ATTENTION_SMOKE_DIR = OUTPUT_ROOT / 'colab_smoke/attention/seed_42'
EXPERIMENT_DIR = OUTPUT_ROOT / 'group_retraining_validation_only'
PREFLIGHT_DIR = EXPERIMENT_DIR / '_preflight'

RUN_FULL_TRAINING = False
CONFIRMATION_TEXT = ''
REQUIRED_CONFIRMATION = 'RUN_40_VALIDATION_ONLY_RUNS'

EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)
PREFLIGHT_DIR.mkdir(parents=True, exist_ok=True)
print({'dataset': str(DATASET_DIR), 'experiment': str(EXPERIMENT_DIR)})

## Sync the repository and preserve Colab CUDA PyTorch

The dependency installer only installs missing non-PyTorch packages. It rejects any plan that would replace Colab's preinstalled CUDA-enabled PyTorch.

In [ ]:
import os
import subprocess
import sys

if (REPO_DIR / '.git').is_dir():
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
CURRENT_COMMIT = subprocess.run(
    ['git', 'rev-parse', 'HEAD'], check=True, capture_output=True, text=True
).stdout.strip()
subprocess.run([
    sys.executable, 'scripts/install_colab_dependencies.py',
    '--requirements', 'requirements-colab.txt',
], check=True)
print('Repository commit:', CURRENT_COMMIT)

## Preflight gates

This stage audits the current CUDA runtime, checks the previously generated leakage audit and validation-only smoke outputs, verifies the fixed 10-second/60-second/30-second pilot dataset metadata, and runs the repository test suite. It reads only audit metadata and smoke artifacts; the full-training commands themselves do not load the test split.

In [ ]:
import json
import torch

subprocess.run([
    sys.executable, 'scripts/audit_colab_environment.py',
    '--repo-dir', str(REPO_DIR),
    '--output', str(PREFLIGHT_DIR / 'environment_audit.json'),
], check=True)
if not torch.cuda.is_available():
    raise RuntimeError('Colab GPU is unavailable. Select Runtime > Change runtime type > GPU.')
print('CUDA GPU:', torch.cuda.get_device_name(0))

test_result = subprocess.run(
    [sys.executable, '-m', 'pytest', '-q'], capture_output=True, text=True
)
test_log = test_result.stdout + test_result.stderr
print(test_log)
(PREFLIGHT_DIR / 'pytest.log').write_text(test_log, encoding='utf-8')
if test_result.returncode != 0:
    raise RuntimeError('Repository tests failed; full training is blocked.')

In [ ]:
from src.colab_workflow import inspect_run_completion

data_audit_path = SETUP_AUDIT_DIR / 'data_artifact_audit.json'
if not data_audit_path.is_file():
    raise FileNotFoundError(
        'Run notebooks/colab_gpu_setup.ipynb first: data audit is missing.'
    )
data_audit = json.loads(data_audit_path.read_text(encoding='utf-8'))
if data_audit.get('case_level_split_integrity') is not True:
    raise RuntimeError('The saved data audit does not confirm disjoint case splits.')

dataset_metadata = json.loads(
    (DATASET_DIR / 'dataset_metadata.json').read_text(encoding='utf-8')
)
expected_pilot = {
    'resampling_interval_seconds': 10,
    'history_window_seconds': 60,
    'history_steps': 6,
    'prediction_horizon_seconds': 30,
}
observed_pilot = {key: dataset_metadata.get(key) for key in expected_pilot}
if observed_pilot != expected_pilot:
    raise RuntimeError(f'Pilot dataset metadata mismatch: {observed_pilot}')

for model_name, run_dir in (
    ('gru', GRU_SMOKE_DIR),
    ('attention', ATTENTION_SMOKE_DIR),
):
    completion = inspect_run_completion(run_dir, model_name)
    if not completion['complete']:
        raise RuntimeError(f'{model_name} CUDA smoke is incomplete: {completion}')
    status = json.loads((run_dir / 'run_status.json').read_text(encoding='utf-8'))
    if status.get('test_evaluated') is not False:
        raise RuntimeError(f'{model_name} smoke did not keep the test split sealed.')

print('Preflight passed:', expected_pilot)

## Fixed experiment plan

All conditions use the same patient-level split, normalized artifacts, optimizer settings, early-stopping rule, CUDA backend, and paired seeds. `bis_error` is excluded in every condition because it is a deterministic duplicate. Candidate comparisons and feature conclusions must use validation outputs only.

In [ ]:
from datetime import datetime, timezone

SEEDS = (7, 21, 42, 84, 123)
MODELS = ('gru', 'attention')
CONDITIONS = {
    'full17': ('bis_error',),
    'no_remifentanil': (
        'bis_error', 'rftn_rate', 'rftn_volume', 'rftn_cp', 'rftn_ce'
    ),
    'no_respiratory': ('bis_error', 'spo2', 'etco2'),
    'no_remifentanil_or_respiratory': (
        'bis_error', 'rftn_rate', 'rftn_volume', 'rftn_cp', 'rftn_ce',
        'spo2', 'etco2',
    ),
}
BATCH_SIZE = 256
MAX_EPOCHS = 50
PATIENCE = 8
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
EXPECTED_RUN_COUNT = len(SEEDS) * len(MODELS) * len(CONDITIONS)
assert EXPECTED_RUN_COUNT == 40

def run_directory(condition_name, model_name, seed):
    return EXPERIMENT_DIR / condition_name / model_name / f'seed_{seed}'

def build_command(condition_name, model_name, seed, resume=None):
    script = 'scripts/run_baselines.py' if model_name == 'gru' else 'scripts/run_attention.py'
    command = [sys.executable, script]
    if model_name == 'gru':
        command.append('gru')
    command.extend([
        '--dataset-dir', str(DATASET_DIR),
        '--output-dir', str(run_directory(condition_name, model_name, seed)),
        '--seed', str(seed),
        '--device', 'cuda',
        '--learning-rate', str(LEARNING_RATE),
        '--weight-decay', str(WEIGHT_DECAY),
        '--batch-size', str(BATCH_SIZE),
        '--max-epochs', str(MAX_EPOCHS),
        '--patience', str(PATIENCE),
        '--num-workers', '0',
        '--exclude-dynamic-features', ','.join(CONDITIONS[condition_name]),
        '--validation-only',
    ])
    if resume is not None:
        command.extend(['--resume', str(resume)])
    return command

plan = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'git_commit_hash': CURRENT_COMMIT,
    'dataset_dir': str(DATASET_DIR),
    'output_dir': str(EXPERIMENT_DIR),
    'validation_only': True,
    'test_split_policy': 'not loaded during candidate training or selection',
    'conditions': {name: list(features) for name, features in CONDITIONS.items()},
    'models': list(MODELS),
    'seeds': list(SEEDS),
    'expected_run_count': EXPECTED_RUN_COUNT,
    'training': {
        'batch_size': BATCH_SIZE,
        'max_epochs': MAX_EPOCHS,
        'patience': PATIENCE,
        'learning_rate': LEARNING_RATE,
        'weight_decay': WEIGHT_DECAY,
        'device': 'cuda',
        'case_balanced_sampling': True,
    },
}
(EXPERIMENT_DIR / 'experiment_plan.json').write_text(
    json.dumps(plan, indent=2, allow_nan=False) + '\n', encoding='utf-8'
)
print(json.dumps(plan, indent=2))

## Deliberate execution gate

Review `experiment_plan.json`, available Drive space, and the Colab runtime before enabling this cell. Set `RUN_FULL_TRAINING = True` and `CONFIRMATION_TEXT = 'RUN_40_VALIDATION_ONLY_RUNS'` in the configuration cell, then rerun from there.

In [ ]:
if not RUN_FULL_TRAINING or CONFIRMATION_TEXT != REQUIRED_CONFIRMATION:
    raise RuntimeError(
        'Full training remains locked. Review the plan and set both execution controls.'
    )
print(f'Execution unlocked for {EXPECTED_RUN_COUNT} validation-only CUDA runs.')

In [ ]:
def read_json(path):
    return json.loads(path.read_text(encoding='utf-8'))

def verify_existing_configuration(run_dir, condition_name, model_name, seed):
    config_path = run_dir / 'config.json'
    if not config_path.exists():
        return
    config = read_json(config_path)
    expected = {
        'seed': seed,
        'device': 'cuda',
        'evaluate_test': False,
        'exclude_dynamic_features': list(CONDITIONS[condition_name]),
        'batch_size': BATCH_SIZE,
        'max_epochs': MAX_EPOCHS,
        'patience': PATIENCE,
    }
    mismatches = {
        key: {'expected': value, 'observed': config.get(key)}
        for key, value in expected.items() if config.get(key) != value
    }
    if config.get('git_commit_hash') != CURRENT_COMMIT:
        mismatches['git_commit_hash'] = {
            'expected': CURRENT_COMMIT, 'observed': config.get('git_commit_hash')
        }
    if mismatches:
        raise RuntimeError(f'Refusing incompatible resume in {run_dir}: {mismatches}')

def verify_validation_only_completion(run_dir, model_name):
    completion = inspect_run_completion(run_dir, model_name)
    if not completion['complete']:
        raise RuntimeError(f'Run did not produce all required artifacts: {completion}')
    status = read_json(run_dir / 'run_status.json')
    config = read_json(run_dir / 'config.json')
    if status.get('test_evaluated') is not False or config.get('evaluate_test') is not False:
        raise RuntimeError(f'Test-seal assertion failed for {run_dir}.')
    forbidden = [
        name for name in ('test_predictions.csv', 'test_metrics.json', 'test_attention.npz')
        if (run_dir / name).exists()
    ]
    if forbidden:
        raise RuntimeError(f'Unexpected test artifacts in {run_dir}: {forbidden}')
    return completion

def run_one(condition_name, model_name, seed):
    run_dir = run_directory(condition_name, model_name, seed)
    run_dir.mkdir(parents=True, exist_ok=True)
    verify_existing_configuration(run_dir, condition_name, model_name, seed)
    completion = inspect_run_completion(run_dir, model_name)
    if completion['complete']:
        verify_validation_only_completion(run_dir, model_name)
        return {'condition': condition_name, 'model': model_name, 'seed': seed, 'action': 'skipped_complete'}

    resume_path = run_dir / 'last_model.pt'
    command = build_command(
        condition_name, model_name, seed,
        resume=resume_path if resume_path.exists() else None,
    )
    started = datetime.now(timezone.utc)
    print('Running:', ' '.join(command))
    result = subprocess.run(command, cwd=REPO_DIR)
    record = {
        'condition': condition_name,
        'model': model_name,
        'seed': seed,
        'action': 'resumed' if '--resume' in command else 'started',
        'return_code': result.returncode,
        'started_at_utc': started.isoformat(),
        'finished_at_utc': datetime.now(timezone.utc).isoformat(),
        'command': command,
    }
    (run_dir / 'colab_execution.json').write_text(
        json.dumps(record, indent=2, allow_nan=False) + '\n', encoding='utf-8'
    )
    if result.returncode != 0:
        raise RuntimeError(f'Training failed in {run_dir}; inspect its status and logs.')
    verify_validation_only_completion(run_dir, model_name)
    return record

In [ ]:
execution_records = []
for condition_name in CONDITIONS:
    for seed in SEEDS:
        for model_name in MODELS:
            execution_records.append(run_one(condition_name, model_name, seed))
            (EXPERIMENT_DIR / 'execution_manifest.json').write_text(
                json.dumps(execution_records, indent=2, allow_nan=False) + '\n',
                encoding='utf-8',
            )
print(f'Completed or verified {len(execution_records)} of {EXPECTED_RUN_COUNT} runs.')

## Validation summary

This summary uses validation metrics only. It is an execution check, not the final statistical comparison or feature-selection decision. Do not evaluate the held-out test set until all candidate decisions are frozen.

In [ ]:
import pandas as pd

rows = []
for condition_name in CONDITIONS:
    for seed in SEEDS:
        for model_name in MODELS:
            run_dir = run_directory(condition_name, model_name, seed)
            verify_validation_only_completion(run_dir, model_name)
            metrics = read_json(run_dir / 'val_metrics.json')
            history = pd.read_csv(run_dir / 'training_history.csv')
            best_row = history.loc[history['validation_patient_level_mae'].idxmin()]
            rows.append({
                'condition': condition_name,
                'model': model_name,
                'seed': seed,
                'dynamic_feature_count': len(read_json(run_dir / 'config.json')['dynamic_feature_names']),
                'best_epoch': int(best_row['epoch']),
                'validation_patient_level_mae': float(best_row['validation_patient_level_mae']),
                'validation_pooled_mae': float(metrics['pooled_window']['regression']['mae']),
                'validation_pooled_rmse': float(metrics['pooled_window']['regression']['rmse']),
            })
summary = pd.DataFrame(rows).sort_values(['condition', 'model', 'seed'])
summary.to_csv(EXPERIMENT_DIR / 'validation_run_summary.csv', index=False)
display(summary)
print('Persistent experiment outputs:', EXPERIMENT_DIR)